In [1]:
import pandas as pd, numpy as np
res=pd.read_csv('/workspace/t3-prism-bo-batch-drop-results.csv')
sug=pd.read_csv('/workspace/t3-prism-bo-suggestions-round1.csv')
# Reconstruct manuscript rebound energy under stated equation
h=1.524; g=9.80665
res['E_stated_mJ']=res.e_rebound_mean*res.mass_g*g*h
res['E_if_velocity_COR_mJ']=res.e_rebound_mean**2*res.mass_g*g*h
print('Reconstructed objectives from campaign CSV')
print(res[['specimen','n_valid','mass_g','t_second_ms_mean','in_dv_ms_mean','e_rebound_mean','E_stated_mJ','E_if_velocity_COR_mJ']].to_string(index=False, float_format=lambda x:f'{x:.4f}'))
print('\nn_valid counts:',res.n_valid.value_counts().sort_index().to_dict())
print('\nRound-2 projection fields')
cols=['trial_index','target_mass_g','mass_printed_g','solid_mass_g','envelope_cm3','envelope_ok','cable_d_print_mm','cable_bridge_ok']
print(sug[cols].to_string(index=False))
print('\nSummary:')
print('target_mass unique',sug.target_mass_g.unique())
print('pred printed mass unique',sug.mass_printed_g.unique())
print('solid mass range',sug.solid_mass_g.min(),sug.solid_mass_g.max())
print('distance from 30.95g range', (sug.solid_mass_g-30.95).min(),(sug.solid_mass_g-30.95).max())
print('screen failures: envelope', (~sug.envelope_ok).sum(),'bridge',(~sug.cable_bridge_ok).sum())
# Per-drop SEM and article floor for t180
res['t_sem_from_sd_99']=res.t180_sd/np.sqrt(99)
res['article_floor_072pct']=0.0072*res.t180_mean
print('\nNoise comparison t180:')
print(res[['specimen','t180_sd','t_sem_from_sd_99','article_floor_072pct']].to_string(index=False,float_format=lambda x:f'{x:.6f}'))
print('median article-floor / per-drop-SEM ratio=',np.nanmedian(res.article_floor_072pct/res.t_sem_from_sd_99))


Reconstructed objectives from campaign CSV
specimen  n_valid  mass_g  t_second_ms_mean  in_dv_ms_mean  e_rebound_mean  E_stated_mJ  E_if_velocity_COR_mJ
  6lhxfy      101 18.5000           55.1786         5.3720          0.0504      13.9285                0.7017
  6nheas      101 21.7300           43.1653         5.2592          0.0402      13.0704                0.5260
  9hhbkp      101 21.6200           23.8982         5.4506          0.0215       6.9474                0.1494
  amdjwm      101     NaN           31.7729         5.2612          0.0296          NaN                   NaN
  autv5r      103 22.0400           29.1661         5.3378          0.0268       8.8309                0.2368
  bag26v      101 21.4200           24.7218         5.0302          0.0241       7.7145                0.1859
  bpx68c      101 20.2300           22.1012         5.3017          0.0204       6.1804                0.1263
  nvxsrv      101 20.6600           28.8580         5.3269          0.0266   

In [2]:
import pandas as pd, numpy as np
from scipy.stats import pearsonr, spearmanr
res=pd.read_csv('/workspace/t3-prism-bo-batch-drop-results.csv')
m=res.dropna(subset=['mass_g','t180_mean','e_rebound_mean']).copy()
print('mapped n=',len(m))
print('t180 < 1:',res.loc[res.t180_mean<1,['specimen','t180_mean']].to_dict('records'))
print('t180 > 1:',int((res.t180_mean>1).sum()),'of',len(res))
print('Pearson mass-t180:',pearsonr(m.mass_g,m.t180_mean))
print('Spearman t180-e:',spearmanr(m.t180_mean,m.e_rebound_mean))
print('range absolute=',res.t180_mean.max()-res.t180_mean.min(), 'relative-to-min %=',100*(res.t180_mean.max()/res.t180_mean.min()-1))
# Check stated mean table rounded E and SD/SEM representation
print('t180 CV range percent',100*(res.t180_sd/res.t180_mean).min(),100*(res.t180_sd/res.t180_mean).max())
# mass noise propagation relative per mapped specimen, using 0.457 g
m['mass_noise_cv_pct']=100*0.457/m.mass_g
print('mass scatter implied E CV % range',m.mass_noise_cv_pct.min(),m.mass_noise_cv_pct.max())


mapped n= 7
t180 < 1: [{'specimen': '6lhxfy', 't180_mean': 0.8930777877843858}, {'specimen': '6nheas', 't180_mean': 0.9970082901461372}, {'specimen': 'amdjwm', 't180_mean': 0.9804953200642736}]
t180 > 1: 5 of 8
Pearson mass-t180: PearsonRResult(statistic=np.float64(0.8291263808685769), pvalue=np.float64(0.021095146051920175))
Spearman t180-e: SignificanceResult(statistic=np.float64(-0.39285714285714296), pvalue=np.float64(0.3833168704269727))
range absolute= 0.1685419860411974 relative-to-min %= 18.872038734646956
t180 CV range percent 0.16939935599812134 0.4822902636326831
mass scatter implied E CV % range 2.0735027223230493 2.4702702702702704
